# 02. Data Pre-Cleaning

Transforms raw data into a pre-cleaned state for downstream processing. Each step logs changes.

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np
import re


In [16]:
raw_dir = Path.cwd().parent.parent / "data" / "raw"


csv_files = list(raw_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Place a CSV in {raw_dir}")

file_path = csv_files[0]
df = pd.read_csv(file_path, encoding="utf-8", low_memory=False)
n_row0, n_col0 = df.shape
print(f"Loaded: {file_path.name}  ({n_row0:,} rows x {n_col0:,} cols)")

Loaded: devices-1681209661.csv  (118,249 rows x 15 cols)


In [43]:
proc_dir = Path.cwd().parent.parent / "data" / "processed"
proc_dir.mkdir(parents=True, exist_ok=True)

## 1. Applying anotations of first EDA

Remove rows where nulls are present.

In [17]:
df.head(1)

,id,classification,code,description,distributed_to,implanted,name,number,quantity_in_commerce,risk_class,slug,country,manufacturer_id,created_at,updated_at
0,1,NaN,NaN,IVD Other,NaN,NaN,Cell-Dyn Emerald Cleanser,NaN,NaN,NaN,tur-cell-dyn-emerald-cleanser,TUR,1,2022-08-04 08:28:00 UTC,2022-08-04 08:28:00 UTC


In [18]:
df = df.dropna()

In [19]:
df.shape

(30556, 15)

In [20]:
df.drop(columns='updated_at', inplace = True)

In [21]:
df.columns

Index(['id', 'classification', 'code', 'description', 'distributed_to',
       'implanted', 'name', 'number', 'quantity_in_commerce', 'risk_class',
       'slug', 'country', 'manufacturer_id', 'created_at'],
      dtype='str')

In [22]:
df.drop(columns='number', inplace = True)

In [23]:
df.columns

Index(['id', 'classification', 'code', 'description', 'distributed_to',
       'implanted', 'name', 'quantity_in_commerce', 'risk_class', 'slug',
       'country', 'manufacturer_id', 'created_at'],
      dtype='str')

In [24]:
df['manufacturer_id']

12529      4827
12530      4828
12531      4828
12532      4828
12533      4828
          ...  
104291     4892
104292    26448
104293     5571
104294     5645
104295     5645
Name: manufacturer_id, Length: 30556, dtype: int64

Unifying two categories in 'risk_class':

In [25]:
df.loc[df['risk_class'] == 'Not Classified', 'risk_class'] = 'Unclassified'

In [26]:
df['risk_class'].unique()

<StringArray>
['2', '1', '3', 'Unclassified', 'HDE']
Length: 5, dtype: str

In [27]:
df['country'].unique()

<StringArray>
['USA']
Length: 1, dtype: str

## 2. Drop constant and duplicate columns

In [28]:
# Constant columns (single value)
const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
df.drop(columns=const_cols, inplace=True)
print(f"Dropped {len(const_cols)} constant column(s): {const_cols}")

Dropped 1 constant column(s): ['country']


In [29]:
# Duplicate columns (identical values)
dup_cols = []
seen = {}
for col in df.columns:
    key = tuple(df[col].astype(str).fillna("__NULL__"))
    if key in seen:
        dup_cols.append(col)
    else:
        seen[key] = col
df.drop(columns=dup_cols, inplace=True)
print(f"Dropped {len(dup_cols)} duplicate column(s): {dup_cols}")

Dropped 0 duplicate column(s): []


## 3. Standardize column names

Convert to snake_case: lowercase, spaces/hyphens to underscores, strip special chars.

In [30]:
def clean_col_name(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"[\s\-]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

old_names = list(df.columns)
new_names = [clean_col_name(c) for c in old_names]
rename_map = {o: n for o, n in zip(old_names, new_names) if o != n}
df.rename(columns=rename_map, inplace=True)
print(f"Renamed {len(rename_map)} column(s)")
if rename_map:
    for old, new in rename_map.items():
        print(f"  {old:30s} -> {new}")

Renamed 0 column(s)


In [31]:
df.columns

Index(['id', 'classification', 'code', 'description', 'distributed_to',
       'implanted', 'name', 'quantity_in_commerce', 'risk_class', 'slug',
       'manufacturer_id', 'created_at'],
      dtype='str')

Columns are normalized

## 4. String cleaning

Strip leading/trailing whitespace from all object columns.

In [33]:
str_cols = df.select_dtypes(include=["object", "string"]).columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip().replace("nan", None)
print(f"Cleaned {len(str_cols)} string column(s)")

Cleaned 10 string column(s)


In [36]:
# Optimize low-cardinality categoricals
cat_cols = df.select_dtypes(include=["object", "string"]).columns
cat_opt = []
for col in cat_cols:
    if df[col].nunique() / len(df) < 0.05:
        df[col] = df[col].astype("category")
        cat_opt.append(col)
print(f"Optimized {len(cat_opt)} column(s) to category: {cat_opt}")

Optimized 3 column(s) to category: ['classification', 'implanted', 'risk_class']


## 6. Deduplicate rows

In [37]:
n_before = len(df)
df.drop_duplicates(inplace=True)
n_dupes = n_before - len(df)
print(f"Removed {n_dupes:,} duplicate row(s) ({n_dupes/n_before*100:.2f}%)")

Removed 0 duplicate row(s) (0.00%)


## 7. Save pre-cleaned data

In [45]:
df.to_csv(f"{proc_dir}/devicesProcessed.csv", index=False, encoding='utf-8')

## 8. Pre-cleaning Report

In [40]:
n_row1, n_col1 = df.shape
print("=" * 55)
print("  PRE-CLEANING REPORT")
print("=" * 55)
print(f"  Original shape:           {n_row0:>8,} rows x {n_col0:,} cols")
print(f"  Final shape:              {n_row1:>8,} rows x {n_col1:,} cols")
print(f"  Rows removed (dupes):     {n_dupes:>8,}")
print(f"  Columns dropped:          {n_col0 - n_col1:>8}")
print(f"    - Constant:             {len(const_cols):>8}")
print(f"    - Duplicate:            {len(dup_cols):>8}")
print(f"  Columns renamed:          {len(rename_map):>8}")
print(f"  Dates parsed:             {len(date_cols):>8}")
print(f"  Optimized to category:    {len(cat_opt):>8}")
print(f"  Remaining missing:        {df.isna().sum().sum():>8,}")

  PRE-CLEANING REPORT
  Original shape:            118,249 rows x 15 cols
  Final shape:                30,556 rows x 12 cols
  Rows removed (dupes):            0
  Columns dropped:                 3
    - Constant:                    1
    - Duplicate:                   0
  Columns renamed:                 0
  Dates parsed:                    0
  Optimized to category:           3
  Remaining missing:               0
